# Prepare Gold Dataset — Resume Parsing (OpenAI vision)

Drafts the gold annotations: each image-only resume PDF is rendered and sent to a vision LLM,
constrained to `parsed_resume_schema.json` (email/phone excluded), validated, and saved as a
draft JSON for human review.

Model `gpt-4o-mini`; a `MAX_RESUMES` cap protects your API credit.

In [13]:
%pip install -q openai pymupdf jsonschema

Note: you may need to restart the kernel to use updated packages.


## Config — set `export OPENAI_API_KEY=sk-...` before launching Jupyter

In [ ]:
import os, json, base64, glob
from openai import OpenAI

MODEL        = "gpt-4o-mini"
MAX_RESUMES  = 5
DPI          = 150
IMG_DETAIL   = "high"
GOLD_JSONL   = "gold.jsonl"
OUT_DIR      = "gold_drafts"
SCHEMA_PATH  = "parsed_resume_schema.json"
EXAMPLE_PATH = "gold_annotation_example.json"

client = OpenAI(api_key="sk")
schema  = json.load(open(SCHEMA_PATH))
example = json.load(open(EXAMPLE_PATH))

## Draft one resume → validated JSON

The system prompt is built from `gold_annotation_example.json` — its `_annotation_guidelines` and worked example drive the extraction, so the documented rules are actually applied.

In [15]:
import fitz
from jsonschema import Draft7Validator

guidelines = "\n".join(f"- {g}" for g in example["_annotation_guidelines"])
worked     = {k: v for k, v in example["examples"][0].items() if not k.startswith("_")}

SYS = (
    "You extract structured data from a resume and return ONLY JSON matching this schema.\n\n"
    "SCHEMA:\n" + json.dumps(schema) + "\n\n"
    "ANNOTATION GUIDELINES:\n" + guidelines + "\n\n"
    "WORKED EXAMPLE (for one resume):\n" + json.dumps(worked)
)

def draft(path):
    doc = fitz.open(path); png = doc[0].get_pixmap(dpi=DPI).tobytes("png"); doc.close()
    b64 = base64.b64encode(png).decode()
    r = client.chat.completions.create(
        model=MODEL, temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role":"system","content":SYS},
                  {"role":"user","content":[
                      {"type":"text","text":"Extract the resume in this image."},
                      {"type":"image_url","image_url":{
                          "url":f"data:image/png;base64,{b64}","detail":IMG_DETAIL}}]}],
    )
    obj = json.loads(r.choices[0].message.content)
    obj["_valid"] = not list(Draft7Validator(schema).iter_errors(obj))
    obj.setdefault("_meta", {})["parser_model"] = MODEL
    return obj

## Run — draft the sampled resumes (exact gold-record format)

In [16]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
CONTENT = ["contact","skills","education","experience","projects","certifications","job_titles"]

gold = [json.loads(l) for l in open(GOLD_JSONL) if l.strip()][:MAX_RESUMES]
print(f"Drafting {len(gold)} resumes from {GOLD_JSONL} -> {OUT_DIR}/\n")

for i, g in enumerate(gold, 1):
    pdf = f"../gold_review/{g['eval_split']}/{g['id']}.pdf"
    if not os.path.exists(pdf):
        print(f"[{i}/{len(gold)}] {g['id']:35s} SKIP (pdf not found: {pdf})")
        continue
    print(f"[{i}/{len(gold)}] {g['id']:35s} parsing...", flush=True)
    d = draft(pdf)
    rec = {"id": g["id"], "category": g["category"], "eval_split": g["eval_split"], "pdf": g["pdf"]}
    for k in CONTENT:
        rec[k] = d.get(k)
    rec["_annotated"] = False
    out = os.path.join(OUT_DIR, g["id"] + ".json")
    json.dump(rec, open(out, "w"), ensure_ascii=False, indent=2)
    print(f"[{i}/{len(gold)}] {g['id']:35s} saved -> {out}  (valid={d.get('_valid')})", flush=True)

print("\ndone.")

Drafting 5 resumes from gold.jsonl -> gold_drafts/

[1/5] sap_developer__Image_100            parsing...
[1/5] sap_developer__Image_100            saved -> gold_drafts/sap_developer__Image_100.json  (valid=True)
[2/5] accountant__fac6c23d5aafc14e        parsing...
[2/5] accountant__fac6c23d5aafc14e        saved -> gold_drafts/accountant__fac6c23d5aafc14e.json  (valid=True)
[3/5] civil_engineer__3e1ceee0957c9c33    parsing...
[3/5] civil_engineer__3e1ceee0957c9c33    saved -> gold_drafts/civil_engineer__3e1ceee0957c9c33.json  (valid=True)
[4/5] building_construction__Image_16     parsing...
[4/5] building_construction__Image_16     saved -> gold_drafts/building_construction__Image_16.json  (valid=True)
[5/5] automobile__e6281a9c220f5215        parsing...
[5/5] automobile__e6281a9c220f5215        saved -> gold_drafts/automobile__e6281a9c220f5215.json  (valid=True)

done.
